# 05 · Clasificación: Bank Marketing

> **Pregunta:** antes de llamar, ¿podemos priorizar a quién contactar para ofrecer un depósito a plazo?

Usaremos datos reales de campañas telefónicas de un banco portugués publicados en UCI. El objetivo es clasificar la etiqueta de suscripción **yes/no**. Una predicción no demuestra que llamar a alguien cause una suscripción.

📊 [UCI Bank Marketing · CC BY 4.0](https://archive.ics.uci.edu/dataset/222/bank+marketing) · DOI 10.24432/C5K306.

## 1. Cargar y conocer los datos

In [ ]:
%pip -q install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

bank = fetch_ucirepo(id=222)
X_original = bank.data.features.copy()
y_original = bank.data.targets.copy()
df = X_original.copy()
df["y"] = y_original.iloc[:, 0].astype(str).str.lower().to_numpy()

print("Dimensiones:", df.shape)
print("Target:", df["y"].value_counts(dropna=False))
print("Tasa positiva:", round((df["y"] == "yes").mean(), 3))
display(df.head())
display(df.dtypes.to_frame("tipo"))

### ¿Está desbalanceada la etiqueta?

Si la mayoría de las personas no suscribe, una regla que siempre predice “no” puede tener accuracy alta y aun así no encontrar a quienes sí suscriben.

In [ ]:
df["y"].value_counts(normalize=True).plot(kind="bar", color=["#176b87", "#b64a36"])
plt.title("Proporción de suscripción")
plt.ylabel("Proporción")
plt.xticks(rotation=0)
plt.show()

## 2. Definir el escenario y evitar fuga

La pregunta es a quién llamar antes de marcar. Excluimos **duration** y **campaign**: la duración se conoce después de la llamada y campaign resume el número total de contactos realizados durante la campaña, por lo que no es una variable segura para anticipar un contacto.

In [ ]:
y = (df["y"] == "yes").astype(int)
X = df.drop(columns=["y", "duration", "campaign"], errors="ignore").copy()

print("Se excluyó duration.")
print("Variables de entrada:", X.shape[1])
print("Tasa de suscripción:", round(y.mean(), 3))
print("Campos tipo texto:", X.select_dtypes(include=["object", "category"]).columns.tolist())

## 3. Interpretar pdays=999

En este dataset, 999 codifica que no hubo contacto previo. No son literalmente 999 días transcurridos. Creamos una bandera y tratamos el sentinel como ausente para imputar el valor numérico usando sólo entrenamiento.

In [ ]:
if "pdays" in X.columns:
    X["not_previously_contacted"] = (X["pdays"] == 999).astype(int)
    X["pdays"] = X["pdays"].replace(999, np.nan)

unknown_counts = {
    col: int((X[col].astype(str).str.lower() == "unknown").sum())
    for col in X.select_dtypes(include=["object", "category"]).columns
}
display(pd.Series(unknown_counts, name="unknown por columna").sort_values(ascending=False).head(10))

## 4. Separar entrenamiento y prueba

Usamos partición estratificada para mantener proporciones de clase parecidas. La semilla fija hace reproducible la partición; no garantiza que un nuevo periodo o banco se comporte igual.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Tasa positiva train:", round(y_train.mean(), 3))
print("Tasa positiva test:", round(y_test.mean(), 3))
print("Tamaños:", X_train.shape, X_test.shape)

## 5. Preparar variables dentro del pipeline

Las variables numéricas se imputan y escalan; las categóricas se imputan y codifican con one-hot. Al agrupar pasos en un pipeline se reduce el riesgo de que las transformaciones aprendan de los datos de prueba.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

categorical = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric = [c for c in X_train.columns if c not in categorical]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

model = Pipeline([
    ("prep", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])
model.fit(X_train, y_train)
print("Modelo ajustado.")

## 6. Comparar con una regla simple

El baseline siempre predice la clase más frecuente. Además de accuracy, observamos precision, recall y F1 para la clase positiva. El balanceo del modelo modifica el ajuste; no determina el umbral apropiado para una campaña.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_test)

prob_yes = model.predict_proba(X_test)[:, 1]
pred_50 = (prob_yes >= 0.50).astype(int)

print("Accuracy baseline:", round(accuracy_score(y_test, pred_base), 3))
print("\nReporte de clasificación, umbral 0.50\n")
print(classification_report(
    y_test, pred_50, target_names=["no suscribe", "suscribe"], zero_division=0
))

## 7. Matriz de confusión y cambio de umbral

Un falso positivo significa priorizar a una persona que no suscribe; un falso negativo deja fuera a alguien que sí habría suscrito. El costo relativo depende del propósito y capacidad del equipo. Aquí comparamos umbrales sólo para aprender cómo cambian las métricas; para escoger uno sin contaminar la prueba, usaremos validación en la notebook 06.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_50, display_labels=["No", "Sí"], cmap="Blues", values_format="d"
)
plt.title("Test · umbral 0.50")
plt.show()

from sklearn.metrics import precision_score, recall_score, f1_score
rows = []
for threshold in [0.30, 0.50, 0.70]:
    predicted = (prob_yes >= threshold).astype(int)
    rows.append({
        "umbral": threshold,
        "precision": precision_score(y_test, predicted, zero_division=0),
        "recall": recall_score(y_test, predicted, zero_division=0),
        "F1": f1_score(y_test, predicted, zero_division=0)
    })
pd.DataFrame(rows).set_index("umbral").round(3)

## 🧠 Interpretación responsable

La probabilidad resume patrones históricos en esta institución y campaña; no es una garantía individual ni una estimación causal del efecto de llamar. No se debe automatizar el contacto sin revisar objetivo, costos, privacidad, sesgos y desempeño en un periodo actual.

## 🧪 Mini reto

Compara precision y recall con umbrales 0.30, 0.50 y 0.70. Si el equipo sólo puede realizar pocas llamadas, ¿qué métrica y política priorizarías? Justifica la decisión y explica qué evidencia adicional necesitarías.

## 🚀 Siguiente

La notebook 06 reserva un test final y compara modelos y umbrales con validación.

## Fuentes y atribución

- Moro, S., Cortez, P. & Rita, P. (2014). UCI Bank Marketing. [Ficha, atribución y licencia CC BY 4.0](https://archive.ics.uci.edu/dataset/222/bank+marketing), DOI 10.24432/C5K306.
- Scikit-learn: [métricas de clasificación](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics) y [pipelines](https://scikit-learn.org/stable/modules/compose.html).